In [ ]:

!pip install -q transformers datasets accelerate


import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "FacebookAI/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.eval()

print("RoBERTa loaded successfully!")


message = "Congratulations! You won $1000! Claim your prize now!"

inputs = tokenizer(
    message,
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=1)
prediction = probabilities.argmax(dim=1).item()

print("Message:", message)
print("Prediction:", "SPAM" if prediction == 1 else "NOT SPAM")
print("Scores:", probabilities.tolist())


from datasets import load_dataset

dataset = load_dataset("ucirvine/sms_spam")

print("Total messages:", len(dataset["train"]))

for message in dataset["train"].select(range(5)):
    print("Message:", message["sms"])
    print("Label:", "SPAM" if message["label"] == 1 else "NOT SPAM")
    print()

from sklearn.model_selection import train_test_split
from datasets import DatasetDict

data = dataset["train"]

# Split into 80% training and 20% remaining
train_idx, remaining_idx = train_test_split(
    range(len(data)),
    test_size=0.20,
    random_state=42,
    stratify=data["label"]
)

# Split remaining data equally into validation and test
val_idx, test_idx = train_test_split(
    remaining_idx,
    test_size=0.50,
    random_state=42,
    stratify=[data[i]["label"] for i in remaining_idx]
)

split_dataset = DatasetDict({
    "train": data.select(train_idx),
    "validation": data.select(val_idx),
    "test": data.select(test_idx)
})

print(split_dataset)


# Convert SMS messages into tokens
def tokenize_messages(examples):
    return tokenizer(
        examples["sms"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = split_dataset.map(
    tokenize_messages,
    batched=True
)

print(tokenized_dataset)


import torch
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("GPU available:", torch.cuda.is_available())

# Measure how well the model identifies spam
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predicted_labels = predictions.argmax(axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predicted_labels,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predicted_labels),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Training settings
training_args = TrainingArguments(
    output_dir="./roberta-spam-baseline",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    seed=42,
    fp16=torch.cuda.is_available()
)

# Create the training pipeline
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Training pipeline ready!")

trainer.train()


# Evaluate on messages the model has never trained on
test_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["test"]
)

print("\nFINAL BASELINE TEST RESULTS")
print("---------------------------")

for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

from google.colab import drive



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa loaded successfully!
Message: Congratulations! You won $1000! Claim your prize now!
Prediction: SPAM
Scores: [[0.4547039270401001, 0.5452960133552551]]
Total messages: 5574
Message: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Label: NOT SPAM

Message: Ok lar... Joking wif u oni...

Label: NOT SPAM

Message: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's

Label: SPAM

Message: U dun say so early hor... U c already then say...

Label: NOT SPAM

Message: Nah I don't think he goes to usf, he lives around here though

Label: NOT SPAM

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 4459
    })
    validation: Dataset({
        features: ['sms', 'label'],
        num_rows: 557
    })
    test: Dataset({
        features: ['sms', 'label'],
        num_rows: 558
    })
}

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.072000,0.079773,0.987433,0.985507,0.918919,0.951049
2,0.015014,0.064850,0.989228,0.972222,0.945946,0.958904


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.015014,0.041376,2,0.994624,0.986486,0.973333,0.979866



FINAL BASELINE TEST RESULTS
---------------------------
eval_loss: 0.0414
eval_accuracy: 0.9946
eval_precision: 0.9865
eval_recall: 0.9733
eval_f1: 0.9799


MessageError: Error: credential propagation was unsuccessful